In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import json
import networkx as nx
import requests
from bs4 import BeautifulSoup
!pip install transformers torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
!pip install accelerate
import accelerate
import os, json, glob

from huggingface_hub import login
login()


# Configure device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f"Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


# Load model once outside the function (so it doesn't reload on every call)
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # use float32 if on CPU
    device_map=torch.device('cuda'),           # automatically uses GPU if availa
)


# intialize pretrained thing, for now just the embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Using device: cuda


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Google drive utils etc. in the event of running in CoLab

In [2]:
from google.colab import drive
drive.mount('/content/drive/')
import os
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CS6540/' #
GOOGLE_DRIVE_PATH = os.path.join('drive', 'MyDrive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

Mounted at /content/drive/
['gstorer-A1-MLP.ipynb', 'gstorer-A1-CNN.ipynb', 'content', 'gstorer-A3-RNN.ipynb', 'WildGraphBench', 'graph.gml', 'threshold_0_6.gml', 'threshold_0_7.gml', 'embeddings.pt', 'transfer_learning.ipynb']


## Fetching, Chunking, and Building the Graph

`fetch_and_chunk` returns a vector of words from parsed from a given url


`build_graph` uses the embedder to create a similarity matrix of all the chunks generated from `fetch_and_chunk`. If entries in the similarity matrix have cosine similarity > `sim_threshold`, then add an edge between the chunks in the graph representation.

In [3]:
REPO_PATH = '/content/drive/MyDrive/CS6540/WildGraphBench'


def load_questions(domain):
    path = f"{REPO_PATH}/QA/{domain}/questions.jsonl"
    with open(path) as f:
        return [json.loads(line) for line in f]


def load_reference_pages(domain, topic, chunk_size=300):
    folder = f"{REPO_PATH}/corpus/{domain}/{topic}/reference_pages/"
    all_chunks = []
    for filepath in glob.glob(folder + "*.txt"):
        with open(filepath, "r", errors="ignore") as f:
            text = f.read()
        words = text.split()
        for i in range(0, len(words), chunk_size):
            chunk = " ".join(words[i:i+chunk_size])
            node_id = f"{os.path.basename(filepath)}__{i}"
            all_chunks.append((node_id, chunk))
    return all_chunks


def build_graph(all_chunks, sim_threshold):
    G = nx.Graph()
    for node_id, chunk in all_chunks:
        G.add_node(node_id, text=chunk)

    embeddings = embedder.encode([c[1] for c in all_chunks], convert_to_tensor=True)
    sim_matrix = util.cos_sim(embeddings, embeddings)

    for i in range(len(all_chunks)):
        for j in range(i + 1, len(all_chunks)):
            if sim_matrix[i][j] > sim_threshold:
                G.add_edge(all_chunks[i][0], all_chunks[j][0], weight=float(sim_matrix[i][j]))

    return G, embeddings

## Retrieval and Responding to Prompts

In [4]:
def retrieve(question, graph, all_chunks, chunk_embeddings, top_k, hopping_number):
    q_emb = embedder.encode([question], convert_to_tensor=True)
    scores = util.cos_sim(q_emb, chunk_embeddings)[0]
    top_results = scores.argsort(descending=True)[:top_k].tolist()
    seed_nodes = [all_chunks[i][0] for i in top_results]
    expanded = set(seed_nodes)
    for node in seed_nodes:
        neighbors = sorted(G[node].items(), key = lambda x: x[1].get("weight", 0), reverse=True)[:hopping_number]
        expanded.update([n for n, _ in neighbors])
    return [graph.nodes[n]["text"] for n in expanded if n in graph.nodes]

def respond(question, context_chunks):
    context = "\n\n".join(context_chunks)
    messages = [
        {"role": "system", "content": "Answer concisely using only the provided context. If the answer isn't in the context, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    # Tokenize using the model's chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict = True,
        truncation = True,
        max_length = 1800
    ).to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,       # greedy decoding — deterministic, good for QA
            temperature=1.0,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (not the prompt)
    generated = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## The Whole Shabang

In [5]:
def graphrag(question, ref_urls):
    G, all_chunks, embeddings = build_graph(ref_urls)
    context = retrieve(question, G, all_chunks, embeddings)
    return respond(question, context)

# Benchmarking

In [9]:
DOMAIN = "culture"
TOPIC = 'Marvel Cinematic Universe'

questions = load_questions(DOMAIN)
all_chunks = load_reference_pages(DOMAIN, TOPIC)
print('checkpoint 1')
G, embeddings = build_graph(all_chunks, sim_threshold = 0.6)
print('checkpoint 2')

checkpoint 1
checkpoint 2


In [ ]:
nx.write_gml(G, "/content/drive/MyDrive/CS6540/threshold_0_6.gml")
torch.save(embeddings, "/content/drive/MyDrive/CS6540/embeddings.pt")

In [10]:
predictions = []
i = 0
for q in questions:
    print("question number ", i)
    i += 1
    context = retrieve(q["question"], G, all_chunks, embeddings, top_k = 3, hopping_number = 2)
    answer = respond(q["question"], context)
    predictions.append({
        "question": q["question"],
        "prediction": answer
    })
    print(f"Q: {q['question']}\nA: {answer}\n")

print('checkpoint 3')
# Save to file
with open("predictions.jsonl", "w") as f:
    for p in predictions:
        f.write(json.dumps(p) + "\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


question number  0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: For the 'A Mini Marvel' commercial, what distinct visual effects processes did Luma Pictures utilize for creating the Hulk versus the technique used for Ant-Man?
A: www.lumapictures.com/careers/) * [Contact](http://www.lumapictures.com/contact/) * [Privacy Policy](http://www.lumapictures.com/privacy-policy/) * [Terms of Use](http://www.lumapictures.com/terms-of-use/)

The provided context is about the development of the Ant-Man and The Wasp: Nano Battle! attraction at Hong Kong Disneyland (HKDL). The attraction is a part of HKDL's multi-year expansion project to become Asia's Marvel hub. The attraction features a heroic battle alongside Ant-Man and The Wasp against Zola and his Swarmbots. The attraction was developed in close partnership with Marvel Studios, with the movie director, leading actors, and composer involved. The attraction also features an immersive, media-rich storytelling, cutting-edge scenic illusion technology, and a state-of-the-art gaming system. Guests are issued

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Q: How did rival studios attempt to replicate the shared universe model popularized by the Marvel Cinematic Universe?
A: that the Hulk, Captain America, Thor, Iron Man, and Black Widow would all be in the same movie? And yet, here they are, in a film that has already grossed more than a billion dollars worldwide.

Marvel’s success has been a long time coming. The company was founded in 1939, but it wasn’t until the nineteen-sixties that it began to publish the comics that would become its most enduring properties. Stan Lee, the company’s editor-in-chief, and Jack Kirby, one of its artists, created Spider-Man, the X-Men, the Fantastic Four, and the Hulk. The characters were a hit, but the company struggled financially. In 1968, Marvel sold the rights to Spider-Man to Columbia Pictures, and the first film, “Spider-Man,” was released in 2002. The film was a hit, and Marvel began to license its characters to other studios. In 2005, Marvel went public, and its stock price soared. The compan

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: What were some common critiques regarding the MCU's interconnected storytelling format during its initial phases?
A: oded onto the scene with 2008's _Iron Man_. But even if you've been following the MCU since the beginning, there's a lot you don't know. That's where _MCU: The Reign of Marvel Studios_ comes in. Written by Joanna Robinson, Dave Gonzales, and Gavin Edwards, the book is an exhaustive, unauthorized look at the rise of Marvel Studios and the Marvel Cinematic Universe. It's a must-read for any fan of the MCU, and it's available now.

# Archived ![Image 2: MCU: The Reign of Marvel Studios cover](https://web.archive.org/web/20230920115931im_/https://www.slashfilm.com/img/gallery/mcu-authors-exclusive-interview/intro-1690821186.jpg) Liveright If you're a longtime /Film reader, you might think you know the history of Marvel Studios. After all, this site has been around since 2005, so in some ways, our archives double as a real-time history of the rise of this improbable pop cu

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: Describe the strategic evolution of Warner Bros.' attempts to build a shared DC cinematic universe and the critical reception of its initial approach.
A: JJ — Thursday November 7, 2013 @ 3:42pm EST * oh yeah those are heavy hitters. Hourman makes Iron Fist look A list lol

DC's cinematic universe was off to a rough start due to rushed production and overzealous film slate announcements before fans had seen more than one film.

Answer: DC's cinematic universe started with a rough start due to hasty production and excessive film slate announcements.

question number  154
Q: What steps has Marvel Studios taken to introduce mutants and X-Men-related concepts into the MCU throughout the Multiverse Saga?
A: ### [Marvel's Kevin Feige Talks MCU's Future, Multiverse Saga, and More at Comic-Con 2022](https://comicbook.com/marvel/news/marvel-kevin-feige-comic-con-2022-interview/)

An incursion is an event in the Marvel Cinematic Universe (MCU) where the boundary between two universes erodes, c